In [33]:
# for reading env file
from dotenv import load_dotenv
import os

# for loading db
import pymysql

# for working part
import numpy as np
import pandas as pd


# for config file 
import json

# for parquet files
import fastparquet

In [2]:
## reading my sql database through env

load_dotenv()



conn = pymysql.connect(
    host=os.getenv("host"),
    port=int(os.getenv("port")),
    user=os.getenv("user"),
    password=os.getenv("password"),
    database=os.getenv("database"),
    ssl={"ssl_mode": "REQUIRED"}
)

cursor = conn.cursor()



print("the db is connected database name is ",os.getenv("database"))

the db is connected database name is  pharma_db


In [3]:
with open("../config/selected_tables.json") as f:
    tables_config = json.load(f)

with open("../config/selected_columns.json") as f:
    columns_config = json.load(f)

In [4]:
def load_tables(selected_tables, columns_config):

    data = {}

    for table in selected_tables:

        cols = ", ".join(columns_config[table])

        query = f"""
        SELECT {cols}
        FROM {table}
        """

        data[table] = pd.read_sql(
            query,
            conn
        )

    return data

In [5]:
selected_tables = tables_config["selected_tables"]

data = load_tables(
    selected_tables,
    columns_config
)

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_19276\1799092350.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(


In [6]:
data.keys()

dict_keys(['calls', 'contacts', 'leads', 'orders', 'pii'])

In [7]:
leads = data["leads"]
contacts = data["contacts"]
pii=data['pii']
calls = data["calls"]
orders = data['orders']

In [8]:
lead_conversion_datset = leads.merge(contacts,how="left",on = "pii_id").merge(pii,how='left',on='pii_id')

lead_conversion_datset

,lead_id,pii_id,owner,lead_source,assigned_date,follow_up_date,contact_id,profile,country,state
0,LD-1,PII-1,None,referral,None,None,NaN,NaN,India,Assam
1,LD-2,PII-2,P-119,special_campaign,2025-04-14,None,CT-1,pharmacy_retailer,India,Karnataka
2,LD-3,PII-3,P-68,app,2025-07-02,None,CT-2,pharmacy_retailer,India,Kerala
3,LD-4,PII-4,P-31,ads,2025-01-09,2025-01-16,NaN,NaN,India,Assam
4,LD-5,PII-5,P-107,special_campaign,2025-02-14,None,CT-3,clinic_owner,India,Maharashtra
...,...,...,...,...,...,...,...,...,...,...
799995,LD-799996,PII-799996,P-50,ads,2025-11-20,None,NaN,NaN,India,West Bengal
799996,LD-799997,PII-799997,P-160,app,2025-08-14,None,CT-513987,online_seller,India,Tamil Nadu
799997,LD-799998,PII-799998,None,app,None,None,NaN,NaN,India,West Bengal
799998,LD-799999,PII-799999,P-71,organic,2025-11-12,None,CT-513988,clinic_owner,India,Gujarat


In [15]:
calls_data = calls.groupby('pii_id').agg(total_duration = ("duration","sum"),call_count = ('call_id',"count"),
                            last_call_date = ("call_date","max"),first_call_date=("call_date","min"),distinct_call_days = ('call_date','nunique'),
                            connected_call_count = ('outcome',lambda x : (x=="connected").sum()),
                            missed_call_count = ('outcome',lambda x : (x=="missed").sum()),
                            inbound_call_count = ('call_type',lambda x: (x=='inbound').sum()),
                            outbound_call_count = ('call_type',lambda x: (x=='outbound').sum()))

In [17]:
lead_conversion_datset = lead_conversion_datset.merge(calls_data,how = "left", on = "pii_id")

In [31]:
date_cols = [
    'assigned_date',
    'follow_up_date',
    'first_call_date',
    'last_call_date'
]

for col in date_cols:
    lead_conversion_datset[col] = pd.to_datetime(
        lead_conversion_datset[col],
        errors='coerce'
    )

In [32]:
lead_conversion_datset.to_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\raw_data.parquet",engine="fastparquet",index=False)